# Nigerian Commodity Price Volatility — Regime-Aware Elastic Net Model

**Objective:** Forecast annualized realized volatility corresponding to Maize, Rice, PMS, and Diesel prices.

## Pipeline Stages
1. **Load Data:** Imports `master_dataset.csv` from Data Engineering phase.
2. **Dimensionality Reduction (PCA):** Compresses highly correlated macro features into orthogonal principal components.
3. **Feature Engineering:** Instantiates $t-1, t-2, t-3$ lagged features and the `Regime_High_Vol` indicator.
4. **Modeling (Elastic Net):** Trains regression with L1/L2 penalties.
5. **Evaluation:** R², RMSE, and Variable Importance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', 50)

print('Libraries imported.')

In [ ]:
# ==========================================
# RUN THIS CELL FIRST IF USING GOOGLE COLAB
# ==========================================
import os
import shutil

try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("✅ Drive mounted successfully.")
    
    if not os.path.exists("output"):
        os.makedirs("output")
        
    !cp "/content/drive/MyDrive/master_dataset.csv" output/
    if os.path.exists("output/master_dataset.csv"):
        print("✅ Successfully loaded master_dataset.csv from Google Drive!")
    else:
        print("⚠️ Copy failed. Does master_dataset.csv exist in the root of your Google Drive?")
        
except ImportError:
    print("ℹ️ Not running in Google Colab, skipping drive mount.")
except Exception as e:
    print(f"❌ Error: {e}")


---
## 1. Load Data
Assumes `master_dataset.csv` is correctly prepared in the `output/` folder.

In [ ]:
import os

# Ensure you are referencing the file created in the datapreparation notebook
file_path = 'output/master_dataset.csv'

if os.path.exists(file_path):
    df_master = pd.read_csv(file_path, parse_dates=['Date'])
    df_master = df_master.set_index('Date').sort_index()
    print(f'Master Dataset loaded: {df_master.shape[0]} rows, {df_master.shape[1]} columns.')
else:
    print(f'⚠️ File not found: {file_path}. Please run datapreparation.ipynb first.')
    df_master = pd.DataFrame()

if not df_master.empty:
    display(df_master.head(3))

---
## 2. Scaling and PCA (Dimensionality Reduction)
We isolate the macroscopic independent variables (Brent Crude, US 10Y Yield, M2, CPI_YoY, Official_FX, Parallel_FX, FX_Premium).
Then apply Standard Scaling and extract Principal Components to alleviate multicollinearity.

In [ ]:
if not df_master.empty:
    # List of expected macroeconomic independent variables
    macro_vars = [
        'Brent_Crude_USD', 'US_10Y_Yield', 'M2_NGN', 
        'Official_FX', 'Parallel_FX', 'FX_Premium', 
        'CPI_YoY', 'Food_CPI_YoY'
    ]
    
    # Ensure we only use columns that actually exist in the dataframe
    available_macro = [c for c in macro_vars if c in df_master.columns]
    print(f'Available macro features for PCA: {available_macro}\n')
    
    df_features = df_master.copy()
    
    # 1. Scale the macroeconomic variables
    scaler = StandardScaler()
    scaled_macro = scaler.fit_transform(df_features[available_macro].fillna(method='bfill').fillna(method='ffill'))
    
    # 2. Fit PCA
    n_components = min(3, len(available_macro))
    pca = PCA(n_components=n_components)
    pca_features = pca.fit_transform(scaled_macro)
    
    print('PCA Explained Variance Ratio:')
    for i, ratio in enumerate(pca.explained_variance_ratio_):
        print(f'  PC{i+1}: {ratio*100:.2f}%')
    print(f'  Total Explained: {sum(pca.explained_variance_ratio_)*100:.2f}%\n')
    
    # 3. Add Principal Components back to the dataframe
    for i in range(n_components):
        df_features[f'Macro_PC{i+1}'] = pca_features[:, i]


---
## 3. Lags and Regime-Aware Feature Engineering
We create lagged versions ($t-1, t-2, t-3$) of our Principal Components to predict future volatility. We define a binary volatility regime based on the `FX_Premium`.

In [ ]:
if not df_master.empty:
    # Generate Lags for our Principal Components
    pca_cols = [f'Macro_PC{i+1}' for i in range(n_components)]
    lags = [1, 2, 3]
    
    for col in pca_cols:
        for lag in lags:
            df_features[f'{col}_lag{lag}'] = df_features[col].shift(lag)
            
    # Define Regime: "High Volatility Regime"
    # E.g., top 25% of the 3-month rolling variance of the FX Premium
    if 'FX_Premium' in df_features.columns:
        fx_roll_var = df_features['FX_Premium'].rolling(3).var()
        threshold = fx_roll_var.quantile(0.75)
        df_features['Regime_High_Vol'] = (fx_roll_var > threshold).astype(int)
        print(f'Defined Regime_High_Vol based on FX Premium variance (threshold: {threshold:.4f})')
        print(f'Number of High Volatility months: {df_features["Regime_High_Vol"].sum()}')
    else:
        print('FX_Premium not found. Defining a dummy regime for testing.')
        df_features['Regime_High_Vol'] = 0
        
    # Drop rows with NaN from lagging
    df_features = df_features.dropna(subset=[f'{col}_lag3' for col in pca_cols])
    print(f'\nFinal feature engineering shape: {df_features.shape}')


---
## 4. Modeling (Elastic Net)
Training the Regime-Aware Elastic Net Model.

In [ ]:
if not df_master.empty:
    # Example predicting Maize Realized Volatility
    target_col = 'Maize_RealVol'
    
    if target_col in df_features.columns:
        # Predictors: The Lagged PCs and the Regime Indicator
        predictors = [f'{col}_lag{lag}' for col in pca_cols for lag in lags] + ['Regime_High_Vol']
        
        # Drop any remaining NAs in target
        model_data = df_features.dropna(subset=[target_col] + predictors)
        
        X = model_data[predictors]
        y = model_data[target_col]
        
        # Chronological Train-Test Split (80% Train, 20% Test)
        split_idx = int(len(X) * 0.8)
        X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
        y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
        
        print(f'Training size: {X_train.shape[0]} months')
        print(f'Testing size: {X_test.shape[0]} months\n')
        
        # Elastic Net CV (Cross-validates to find best L1 ratio and alpha penalty)
        regr = ElasticNetCV(cv=5, l1_ratio=[.1, .5, .7, .9, .95, .99, 1], random_state=42)
        regr.fit(X_train, y_train)
        
        print(f'Optimal Alpha (Penalty): {regr.alpha_:.4f}')
        print(f'Optimal L1 Ratio (1=Lasso, 0=Ridge): {regr.l1_ratio_}')
        
        # Evaluation
        preds = regr.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        mae = mean_absolute_error(y_test, preds)
        r2 = r2_score(y_test, preds)
        
        print(f'\n--- Testing Performance ---')
        print(f'RMSE: {rmse:.4f}')
        print(f'MAE:  {mae:.4f}')
        print(f'R²:   {r2:.4f}')
    else:
        print(f'Target {target_col} not found in the dataset.')


---
## 5. Economic Interpretation
Extracting the coefficients driven to zero vs. kept by the Elastic Net Model.

In [ ]:
if not df_master.empty and target_col in df_features.columns:
    coef_df = pd.DataFrame({
        'Feature': X.columns,
        'Coefficient': regr.coef_
    })
    
    # Filter out features driven exactly to 0 by Lasso
    coef_df = coef_df[coef_df['Coefficient'] != 0].sort_values(by='Coefficient', key=abs, ascending=False)
    
    print('Non-Zero Drivers of Volatility (Elastic Net Coefficients):')
    display(coef_df)
    
    # Plotting actual vs predicted on test set
    plt.figure(figsize=(10, 5))
    plt.plot(y_test.index, y_test.values, label='Actual Volatility', marker='o')
    plt.plot(y_test.index, preds, label='Predicted Volatility', marker='x', linestyle='--')
    plt.title(f'{target_col} - Actual vs Predicted (Test Set)')
    plt.ylabel('Realized Volatility')
    plt.legend()
    plt.show()
